# Making SIMSOPT GPU native: local-residual parity

Select **Runtime > Change runtime type > GPU**, then run all cells. This experiment qualifies physically normalized, unsquared local engineering residuals before they are used by another optimizer. It compares CPU/GPU values at production thresholds and CPU finite-difference directional derivatives against GPU JVPs at a deterministic off-symmetry probe.

In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "gpu-native-objective", "https://github.com/PedroFranciscoGil/simsopt.git", str(repo)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "pytest"], cwd=repo, check=True)
os.chdir(repo)
source_root = repo / "src"
sys.path.insert(0, str(source_root))
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(revision)

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
print(f"Imported SIMSOPT from {resolved_package}")
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report
assert jax.config.jax_enable_x64, report
assert any(device.platform == "gpu" for device in jax.devices()), report

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/gpu/test_distances.py", "tests/gpu/test_regularizers.py", "tests/gpu/test_objective.py", "tests/gpu/test_local_residual_benchmark.py"], cwd=repo, check=True)

In [ ]:
artifact_root = Path("/content/simsopt-local-residual-parity")
artifact_root.mkdir(exist_ok=True)
result_path = artifact_root / "local-residual-parity.json"
env = os.environ.copy()
env["OMP_NUM_THREADS"] = "1"
env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
subprocess.run([sys.executable, "benchmarks/gpu/benchmark_local_residual_parity.py", "--problem", "engineering", "--directions", "3", "--finite-difference-step", "1e-7", "--repeats", "7", "--current-scale", "100000", "--target-tile-size", "1024", "--source-tile-size", "4320", "--output", str(result_path)], cwd=repo, env=env, check=True)

In [ ]:
import json

result = json.loads(result_path.read_text())
assert result["schema_version"] == 1
assert result["workflow"] == "local_engineering_residual_parity"
assert result["environment"]["jax_backend"] == "gpu"
assert result["precision"] == "float64"
assert result["residual_contract"]["family_order"] == ["coil_coil_distance", "coil_surface_distance", "curvature", "mean_squared_curvature"]
failed = {name: gate for name, gate in result["acceptance_gates"].items() if not gate["passed"]}
print(json.dumps({"all_gates_passed": result["all_gates_passed"], "failed_gates": failed, "timing": result["timing"]}, indent=2))

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/simsopt-local-residual-parity", "zip", artifact_root)
files.download(archive)

## What to send back

Send the downloaded `simsopt-local-residual-parity.zip`. A failed scientific gate is still useful data; the archive is downloaded without asserting that every gate passed.